In [37]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
import joblib
from xgboost import XGBClassifier


In [39]:
df = pd.read_csv("combined_dataset.csv")

# 2. Define features and label
features = [
    "Nice", "Priority", "Threads", "VmSize_KB", "VmRSS_KB",
    "text_size", "data_size", "avg_cpu", "avg_memory", "avg_duration"
]
X = df[features]
y = df["class"]

# 3. Encode string labels into integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [40]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)


In [41]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [42]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

print("📊 Random Forest Validation Report:")
print(classification_report(label_encoder.inverse_transform(y_val),
                            label_encoder.inverse_transform(rf_model.predict(X_val_scaled))))

print("📊 Random Forest Test Report:")
print(classification_report(label_encoder.inverse_transform(y_test),
                            label_encoder.inverse_transform(rf_model.predict(X_test_scaled))))


📊 Random Forest Validation Report:
                    precision    recall  f1-score   support

        Background       0.95      0.95      0.95      2685
         Real-Time       0.95      0.94      0.94      2693
Resource-Intensive       0.97      0.98      0.98      4622

          accuracy                           0.96     10000
         macro avg       0.96      0.96      0.96     10000
      weighted avg       0.96      0.96      0.96     10000

📊 Random Forest Test Report:
                    precision    recall  f1-score   support

        Background       0.95      0.96      0.96      2686
         Real-Time       0.96      0.93      0.94      2692
Resource-Intensive       0.97      0.98      0.98      4622

          accuracy                           0.96     10000
         macro avg       0.96      0.96      0.96     10000
      weighted avg       0.96      0.96      0.96     10000



In [43]:
xgb_model = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train)

print("📊 XGBoost Validation Report:")
print(classification_report(label_encoder.inverse_transform(y_val),
                            label_encoder.inverse_transform(xgb_model.predict(X_val_scaled))))

print("📊 XGBoost Test Report:")
print(classification_report(label_encoder.inverse_transform(y_test),
                            label_encoder.inverse_transform(xgb_model.predict(X_test_scaled))))


c:\Users\kalan\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:03:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


📊 XGBoost Validation Report:
                    precision    recall  f1-score   support

        Background       0.95      0.95      0.95      2685
         Real-Time       0.95      0.93      0.94      2693
Resource-Intensive       0.97      0.98      0.98      4622

          accuracy                           0.96     10000
         macro avg       0.96      0.96      0.96     10000
      weighted avg       0.96      0.96      0.96     10000

📊 XGBoost Test Report:
                    precision    recall  f1-score   support

        Background       0.95      0.96      0.96      2686
         Real-Time       0.96      0.93      0.95      2692
Resource-Intensive       0.97      0.98      0.98      4622

          accuracy                           0.96     10000
         macro avg       0.96      0.96      0.96     10000
      weighted avg       0.96      0.96      0.96     10000



In [45]:
joblib.dump(xgb_model, "task_classifier_xgb.joblib")
joblib.dump(rf_model, "task_classifier_rf.joblib")
joblib.dump(scaler, "task_scaler.joblib")
joblib.dump(label_encoder, "task_label_encoder.joblib")


['task_label_encoder.joblib']